Objetos del juego buscaminas:


*   Player
*   Board


In [ ]:
import numpy as np
import random
from google.colab import output

Para self.board:
* -1 = Mina
* 0 = No minas
* 1 = Una mina adyacente
* así sucesivamente...


Para self.visibility:
* -1 = Bandera
* 0 = No visible
* 1 = Visible

In [ ]:
class Board:

  graphing_emojis = {
      -1: "💣",
      0: "⬜️",
      1: "1️⃣",
      2: "2️⃣",
      3: "3️⃣",
      4: "4️⃣",
      5: "5️⃣",
      6: "6️⃣",
      7: "7️⃣",
      8: "8️⃣"
  }

  def __init__(self, rows, columns):
    self.rows = rows
    self.columns = columns
    self.board = np.zeros((rows, columns))
    self.visibility = np.zeros((rows, columns))
    self.n_mines = 0

  def check_win(self):
    target = self.rows * self.columns - self.n_mines
    return np.count_nonzero(self.visibility == 1) == target

  def set_mine(self, row, column):
    self.board[row, column] = -1

  def toggle_flag(self, row, column):
    if self.has_flag(row, column):
      self.visibility[row, column] = 0
    elif self.visibility[row, column] != 1:
      self.visibility[row, column] = -1

  def discover_cell(self, row, column, force_clean=False):
    self.visibility[row, column] = 1
    if force_clean:
      if self.has_mine(row, column):
        self.board[row, column] = 0
        self.__assign_numbers()
        self.n_mines -= 1
    if self.board[row, column] == 0:
      self.__discover_consecutive_blank_cells(row, column)

  def has_mine(self, row, column):
    return self.board[row, column] == -1

  def has_flag(self, row, column):
    return self.visibility[row, column] == -1

  @staticmethod
  def build_board(rows, columns, n_mines=10):
    board = Board(rows, columns)
    n_mines_placed = 0
    while n_mines_placed < n_mines:
      row = random.randint(0, rows-1)
      column = random.randint(0, columns-1)
      if not board.has_mine(row, column):
        board.set_mine(row, column)
        n_mines_placed += 1
    board.__assign_numbers()
    board.n_mines = n_mines_placed
    return board

  def __count_mines_in_range(self, row, column, radius=1):
    n_mines = 0
    for i in range(max(0, row-radius), min(self.rows, row+radius+1)):
      for j in range(max(0, column-radius), min(self.columns, column+radius+1)):
        if self.has_mine(i, j):
          n_mines += 1
    return n_mines

  def __discover_mines_in_range(self, row, column, radius=1):
    adjacent_blank_cells = []
    for i in range(max(0, row-radius), min(self.rows, row+radius+1)):
      for j in range(max(0, column-radius), min(self.columns, column+radius+1)):
        self.visibility[i, j] = 1
        if self.board[i, j] == 0:
          adjacent_blank_cells.append((i, j))
    return adjacent_blank_cells

  def __discover_consecutive_blank_cells(self, row, column):
    # Pseudo-DFS-esque algo
    queue = [(row, column)]
    explored = set()
    while queue:
      cell_coords = queue.pop(0)
      explored.add(cell_coords)
      current_row, current_column = cell_coords
      adjacent_blank_cells = self.__discover_mines_in_range(current_row, current_column)
      for cell in adjacent_blank_cells:
        if cell not in explored:
          queue.append(cell)

  def __assign_numbers(self):
    for i in range(self.rows):
      for j in range(self.columns):
        if self.board[i, j] != -1:
          self.board[i, j] = self.__count_mines_in_range(i, j)

  def __str__(self):
    board_string = ""
    for i in range(self.rows):
      for j in range(self.columns):
        if self.visibility[i, j] == 0:
          board_string += "⬛ "
        elif self.visibility[i, j] == -1:
          board_string += "🚩 "
        else:
          board_string += self.graphing_emojis[int(self.board[i, j])] + " "
      board_string += "\n"
    return board_string

  def __repr__(self):
    return self.__str__()

In [ ]:
class Player:
  def __init__(self, name):
    self.name = name

In [ ]:
class Game:

  def __init__(self, board_size=(10, 10), n_mines=10, player_name="Player"):
    self.board = Board.build_board(board_size[0], board_size[1], n_mines=n_mines)
    self.player = Player(player_name)
    self.turn = 0

  def start(self):
    self.main_loop()

  def main_loop(self):

    while True:

      action = None
      while not action in ["1", "2"]:
        output.clear()
        print(self.board)
        print("Elija la acción a realizar:")
        print("1. Poner bandera.")
        print("2. Descubrir celda.")
        action = input("Acción: ")

      while True:
        output.clear()
        print(self.board)
        print("Elija la celda a trabajar:")
        try:
          row, column = map(int, input("Coordenada: ").split(","))
          if row < 0 or row >= self.board.rows or column < 0 or column >= self.board.columns:
            print("Coordenada inválida.")
          else:
            break
        except AttributeError:
          print("Coordenada inválida.")

      if action == "1":
        self.board.toggle_flag(row, column)
      else:
        force_clean = (self.turn == 0)
        self.board.discover_cell(row, column, force_clean=force_clean)
        if self.board.has_mine(row, column):
          output.clear()
          print(self.board)
          print("BOOM!")
          break
        if self.board.check_win():
          output.clear()
          print(self.board)
          print("You won!")
          break

      self.turn += 1

In [ ]:
game = Game(board_size=(10, 10), n_mines=10)
game.start()

⬜️ ⬜️ ⬜️ ⬜️ ⬜️ ⬜️ 1️⃣ 🚩 1️⃣ ⬜️ 
⬜️ 1️⃣ 1️⃣ 1️⃣ ⬜️ ⬜️ 1️⃣ 1️⃣ 1️⃣ ⬜️ 
⬜️ 1️⃣ 🚩 1️⃣ ⬜️ 1️⃣ 1️⃣ 1️⃣ ⬜️ ⬜️ 
1️⃣ 2️⃣ 2️⃣ 1️⃣ ⬜️ 1️⃣ 🚩 1️⃣ ⬜️ ⬜️ 
1️⃣ 🚩 2️⃣ 1️⃣ ⬜️ 1️⃣ 1️⃣ 1️⃣ 1️⃣ 1️⃣ 
1️⃣ 2️⃣ 🚩 1️⃣ ⬜️ ⬜️ ⬜️ 1️⃣ 3️⃣ 🚩 
⬜️ 2️⃣ 2️⃣ 2️⃣ ⬜️ ⬜️ ⬜️ 1️⃣ 🚩 🚩 
⬜️ 1️⃣ 🚩 1️⃣ ⬜️ ⬜️ ⬜️ 2️⃣ 3️⃣ 3️⃣ 
⬜️ 1️⃣ 1️⃣ 1️⃣ ⬜️ ⬜️ ⬜️ 1️⃣ 🚩 1️⃣ 
⬜️ ⬜️ ⬜️ ⬜️ ⬜️ ⬜️ ⬜️ 1️⃣ 1️⃣ 1️⃣ 

You won!
